Simple critic LLM use case 
HumanMessage  -> Gemini LLM -> Result

Result -> Groq LLM (critic) -> Additional info and critic as per its Knowledge

In [68]:
#imports
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.messages import HumanMessage,BaseMessage,SystemMessage
from langgraph.graph import StateGraph
from typing import List,TypedDict,Annotated

In [69]:

#Setting up Envs
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_LLM_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_LLM_API_KEY")

In [70]:
# Setting Up Logger
import sys
import os

# Get the path to the parent directory
parent_dir = os.path.abspath("..")

# Add it to the search path if it's not already there
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    
from RSYLogger.Logger import Logger
logger = Logger("1 Simple Reflection Pattern using LLM")

In [71]:
#llm creation
logger.info("\nINITIALIZING LLMS")
try:
    logger.info("INITIALIZING LLM : Gemini")
    # gemini_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest",google_api_key=GEMINI_API_KEY)
    gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7,google_api_key=GEMINI_API_KEY)
except Exception as e:
    logger.info("Failed To initialize llm : Gemini")
try:
    logger.info("INITIALIZING LLM : Groq")
    groq_llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7,api_key=GROQ_API_KEY)
except Exception as e:
    logger.info("Failed To initialize llm : Groq")

logger.info("INITIALIZING LLMS : Complete")


In [72]:
#Defining State

class GraphState(TypedDict):
    messages:Annotated[List[BaseMessage],lambda x,y:x+y]

In [73]:
def call_gemini_llm(state:GraphState):
    logger.info("Invoking Gemini LLM")
    logger.info(f"current state:{state}")
    logger.info("Adding System Prompt")
    system_instruction = SystemMessage(
        content="You are a helpful assistant. Keep all your responses extremely short, concise, and brief."
    )
    messages = [system_instruction] + state["messages"]
    logger.info(f"Updated State:{messages}")
    response = gemini_llm.invoke(messages)
    return {"messages":[response]}

In [74]:
def call_groq_llm(state:GraphState):
    logger.info("Invoking Groq LLM")
    logger.info(f"current state:{state}")
    messages = state.get("messages")
    response = groq_llm.invoke(messages)
    return {"messages":[response]}

In [75]:
def fetch_information(state:GraphState):
    logger.info("Fetching Infomation")
    logger.info(f"Current State:{state}")
    response = call_gemini_llm(state)
    return response

In [76]:
def validate_information(state:GraphState):
    logger.info("Validating Infomation")
    logger.info(f"Current State:{state}")
    
    history = state.get("messages")
    logger.info("Generating System Prompt")
    critic_prompt = SystemMessage(content=(
        "You are an expert critic. Look at the last AI response."
        "Keep all your responses extremely short, concise, and brief."
        "Identify any inaccuracies, missing details, or tone issues. "
        "Provide a list of improvements."
        "Add prefix : Let me think what you said before every response"
        "Then mention the response provided at last AI response e.g. Ok so you said <response>"
    ))
    newState = {"messages":[critic_prompt] + history}
    logger.info(f"Updated State:{newState}")
    response = call_groq_llm(newState)
    return response

In [77]:
#WorkFlow Setup
logger.info("Workflow SetUp : Initialized")
workflow = StateGraph(GraphState)
workflow.add_node("information_fetch_gemini",fetch_information)
workflow.add_node("information_validate_groq",validate_information)
workflow.set_entry_point("information_fetch_gemini")
workflow.set_finish_point("information_validate_groq")
workflow.add_edge("information_fetch_gemini","information_validate_groq")

app = workflow.compile()
logger.info("Workflow SetUp : Completed")




In [78]:
def process(prompt=""):
    prompt = HumanMessage(content=prompt)

    inputs = {
        'messages':[prompt]
    }

    for output in app.stream(inputs):
        for key,value in output.items():
            logger.info(f"Output from node '{key}':")
            logger.info(value)

    results = app.invoke(inputs)

    for message in results["messages"]:
     logger.info(f'{message.type} : {message.content}')
     print(f'{message.type} : {message.content}')

In [79]:
process("If I have 3 apples and you take away 2, how many apples do I have left? Explain why in detail.")

human : If I have 3 apples and you take away 2, how many apples do I have left? Explain why in detail.
ai : You have 1 apple left.

**Explanation:**
You begin with an initial quantity of 3 apples. The action "take away 2" means that 2 apples are removed from your starting amount. This is represented by the mathematical operation of subtraction. Therefore, you calculate 3 (your starting apples) minus 2 (apples taken away), which equals 1.
ai :  

**Improvements:**
1. Provide a step-by-step calculation.
2. Clarify the mathematical operation used.
3. Use simpler language for easier understanding.


In [80]:
# from google import genai

# client = genai.Client(api_key=GEMINI_API_KEY)

# for model in client.models.list():
#     print(model.name)